# fast_array_utils

## Example 1: sums and means of a dense array

In [ ]:
# Example 1: stats.sum / mean along an axis or over everything, matching NumPy
import numpy as np
from fast_array_utils import stats

x = np.arange(12, dtype=np.float64).reshape(3, 4)
assert np.allclose(stats.sum(x, axis=0), x.sum(axis=0)) and stats.sum(x) == x.sum()
assert np.allclose(stats.mean(x, axis=1), x.mean(axis=1))
print("Example 1:", stats.sum(x, axis=0), stats.mean(x, axis=1))

## Example 2: the same on a SciPy sparse matrix

In [ ]:
# Example 2: sparse CSR input gives the same answers without densifying
import numpy as np
import scipy.sparse as sp
from fast_array_utils import stats

dense = np.array([[0, 2, 0], [1, 0, 3], [0, 0, 4]], dtype=np.float32)
csr = sp.csr_array(dense)
assert np.allclose(stats.sum(csr, axis=0), dense.sum(axis=0))
assert np.allclose(stats.mean(csr, axis=1), dense.mean(axis=1))
print("Example 2:", np.asarray(stats.sum(csr, axis=0)))

## Example 3: mean and variance in one pass

In [ ]:
# Example 3: mean_var matches NumPy's mean and variance (ddof=1 here)
import numpy as np
import scipy.sparse as sp
from fast_array_utils import stats

rng = np.random.default_rng(0)
x = rng.poisson(1.0, size=(200, 5)).astype(np.float64)
mean, var = stats.mean_var(sp.csr_array(x), axis=0, correction=1)
assert np.allclose(mean, x.mean(axis=0)) and np.allclose(var, x.var(axis=0, ddof=1)), (mean, var)
print("Example 3:", np.round(mean, 3), np.round(var, 3))

## Example 4: is_constant along an axis

In [ ]:
# Example 4: is_constant finds constant rows and columns (a numba kernel for sparse input)
import numpy as np
import scipy.sparse as sp
from fast_array_utils import stats

x = np.array([[1, 1, 1], [0, 5, 0], [2, 2, 2]], dtype=np.float64)
assert stats.is_constant(x, axis=1).tolist() == [True, False, True]
assert stats.is_constant(sp.csr_array(x), axis=1).tolist() == [True, False, True]
assert stats.is_constant(np.full((2, 2), 3.0))
print("Example 4:", stats.is_constant(x, axis=1).tolist())

## Example 5: dense conversion and Dask

In [ ]:
# Example 5: conv.to_dense turns sparse and Dask arrays into NumPy; stats work on Dask lazily
import dask.array as da
import numpy as np
import scipy.sparse as sp
from fast_array_utils import conv, stats

dense = conv.to_dense(sp.csr_array(np.eye(3)))
assert isinstance(dense, np.ndarray) and dense.trace() == 3
lazy = da.from_array(np.arange(16.0).reshape(4, 4), chunks=2)
total = stats.sum(lazy, axis=0)
assert np.allclose(conv.to_dense(total, to_cpu_memory=True) if hasattr(total, "compute") else total, np.arange(16.0).reshape(4, 4).sum(axis=0))
print("Example 5:", type(dense).__name__, type(total).__name__)